# Wan 2.1 SteadyDancer — Перенос танца

**Перенос движений из видео с танцем на любого персонажа.** Загрузите видео танца + фото человека → получите видео, где этот человек танцует.

**Время настройки:** ~15-20 мин (скачивание моделей ~20 ГБ)
**VRAM:** ~15 ГБ пиковое потребление (T4 16 ГБ — на пределе!)

---

### Как это работает

```
[1. GPU]  →  [2. Установка]  →  [3. Ноды]  →  [4. Модели]  →  [5. Воркфлоу]  →  [6. ЗАГРУЗКА]  →  [7. Запуск]
                                                                                        ↑
                                                                               Два файла нужны:
                                                                          1. Видео с танцем (.mp4)
                                                                          2. Фото персонажа (.jpg/.png)
```

---

### Что делает этот ноутбук?

```
Видео с танцем  ──────────┐
                          ├──→  [Детекция поз]  ──→  [SteadyDancer]  ──→  Видео с вашим персонажем
Фото персонажа  ──────────┘       (ViTPose)           (Wan 2.1)           танцующим тот же танец
```

| Этап | Описание |
|------|----------|
| **Детекция поз** | ViTPose + YOLOv10 извлекают скелет из видео с танцем |
| **Кодирование** | CLIP Vision + VAE кодируют фото персонажа |
| **Генерация** | SteadyDancer генерирует персонажа в позах из танца |
| **Выход** | H.264 MP4 видео 720x1280 @ 30fps со звуком из оригинала |

---

### Что загружать?

| Файл | Формат | Требования |
|------|--------|------------|
| **Видео с танцем** | .mp4 | Полная фигура, стабильная камера, 5-15 сек |
| **Фото персонажа** | .jpg, .png | В полный рост или по пояс, чёткое, без водяных знаков |

### Чего НЕ нужно

- Видео с несколькими людьми (только 1 человек)
- Фото с закрытыми частями тела (маски, громоздкая одежда)
- Видео дольше 15 секунд (на T4 может не хватить памяти)

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu}")
    print(f"VRAM: {vram:.1f} ГБ")
    if vram < 15:
        print("\nSteadyDancer требует ~15 ГБ VRAM. Может быть тесно!")
    else:
        print("\nVRAM достаточно для SteadyDancer.")

In [ ]:
#@title 2. Установка ComfyUI
%cd /content
!test -d ComfyUI || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

# Фиксируем numpy < 2.0
!echo "numpy<2.0" > /tmp/numpy_constraint.txt
!pip install "numpy>=1.26,<2.0" -q
!pip install -r requirements.txt -q -c /tmp/numpy_constraint.txt

print("\nComfyUI установлен!")

In [ ]:
#@title 3. Установка кастомных нод
%cd /content/ComfyUI/custom_nodes

repos = {
    "ComfyUI-WanVideoWrapper": "https://github.com/kijai/ComfyUI-WanVideoWrapper.git",
    "ComfyUI-WanAnimatePreprocess": "https://github.com/kijai/ComfyUI-WanAnimatePreprocess.git",
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "ComfyUI-VideoHelperSuite": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "ComfyUI-Easy-Use": "https://github.com/yolain/ComfyUI-Easy-Use.git",
}

for name, url in repos.items():
    !test -d {name} || git clone {url}

# ONNX Runtime GPU — устанавливаем ДО requirements нод, чтобы избежать конфликта с CPU-версией
!pip install onnxruntime-gpu -q -c /tmp/numpy_constraint.txt

# Зависимости
import os
for name in repos:
    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"
    if os.path.exists(req):
        !pip install -r {req} -q -c /tmp/numpy_constraint.txt

print(f"\n{len(repos)} кастомных нод установлено!")

In [ ]:
#@title 4. Скачивание моделей (~15 ГБ, 10-15 мин)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)
os.makedirs(f"{M}/clip_vision", exist_ok=True)
os.makedirs(f"{M}/loras", exist_ok=True)
os.makedirs(f"{M}/detection", exist_ok=True)

KIJAI = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main"
KIJAI_FP8 = "https://huggingface.co/Kijai/WanVideo_comfy_fp8_scaled/resolve/main"
COMFY = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files"

files = {
    # SteadyDancer fp8 (~14.5 ГБ на диске, ~8 ГБ в VRAM с block swap)
    f"{M}/diffusion_models/Wan21_SteadyDancer_fp8_e4m3fn_scaled_KJ.safetensors":
        f"{KIJAI_FP8}/SteadyDancer/Wan21_SteadyDancer_fp8_e4m3fn_scaled_KJ.safetensors",
    # Текстовый энкодер UMT5 XXL fp8 scaled (~6.3 ГБ вместо ~10.6 ГБ bf16)
    f"{M}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors":
        f"{COMFY}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    # VAE bf16 (~400 МБ)
    f"{M}/vae/Wan2_1_VAE_bf16.safetensors":
        f"{KIJAI}/Wan2_1_VAE_bf16.safetensors",
    # CLIP Vision H (~1.3 ГБ)
    f"{M}/clip_vision/clip_vision_h.safetensors":
        f"{COMFY}/clip_vision/clip_vision_h.safetensors",
    # LightX2V LoRA для 4-шаговой генерации (~738 МБ)
    f"{M}/loras/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors":
        f"{KIJAI}/Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors",
    # ViTPose ONNX для детекции поз (~1.2 ГБ)
    f"{M}/detection/vitpose-l-wholebody.onnx":
        "https://huggingface.co/JunkyByte/easy_ViTPose/resolve/main/onnx/wholebody/vitpose-l-wholebody.onnx",
    # YOLOv10m ONNX для детекции людей (~62 МБ)
    f"{M}/detection/yolov10m.onnx":
        "https://huggingface.co/Wan-AI/Wan2.2-Animate-14B/resolve/main/process_checkpoint/det/yolov10m.onnx",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

# Валидация скачивания
for dest in files:
    if os.path.exists(dest) and os.path.getsize(dest) < 1024:
        os.remove(dest)
        print(f"  ОШИБКА: {os.path.basename(dest)} не скачан — перезапустите ячейку")

print("\nВсе модели готовы!")
!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/vae/* {M}/clip_vision/* {M}/loras/* {M}/detection/*

In [ ]:
#@title 5. Скачивание воркфлоу
import os, json

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

!wget -q -O "{WF_DIR}/video_wan_dancer.json" "{RAW}/video_wan_dancer.json"
print("Скачано: video_wan_dancer.json")

# Патчим воркфлоу: заменяем bf16 текстовый энкодер на fp8
wf_path = f"{WF_DIR}/video_wan_dancer.json"
with open(wf_path) as f:
    wf = json.load(f)
patched = False
for node in wf["nodes"]:
    wvals = node.get("widgets_values", [])
    for i, v in enumerate(wvals):
        if isinstance(v, str) and "umt5" in v and v != "umt5_xxl_fp8_e4m3fn_scaled.safetensors":
            wvals[i] = "umt5_xxl_fp8_e4m3fn_scaled.safetensors"
            patched = True
if patched:
    with open(wf_path, "w") as f:
        json.dump(wf, f)
    print("Воркфлоу обновлён: текстовый энкодер -> fp8")

print(f"\nВоркфлоу сохранён в {WF_DIR}")

In [ ]:
#@title 6. Загрузка медиа (видео танца + фото персонажа)
#@markdown ## Загрузите 2 файла:
#@markdown 1. **Видео с танцем** (.mp4) — полная фигура, 1 человек, 5-15 сек
#@markdown 2. **Фото персонажа** (.jpg/.png) — в полный рост или по пояс

from google.colab import files
from IPython.display import display, Image as IPImage
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

VALID_IMAGES = ('.jpg', '.jpeg', '.png')
VALID_VIDEO = ('.mp4',)

print("=" * 60)
print("  ЗАГРУЗКА МЕДИА ДЛЯ STEADYDANCER")
print("=" * 60)
print(f"  Папка: {INPUT_DIR}")
print()
print("  Нужны 2 файла:")
print("    1. Видео с танцем (.mp4)")
print("    2. Фото персонажа (.jpg / .png)")
print()

uploaded = files.upload()

images = []
videos = []
skipped = []

for name, data in uploaded.items():
    ext = os.path.splitext(name)[1].lower()
    path = os.path.join(INPUT_DIR, name)
    if ext in VALID_IMAGES:
        with open(path, "wb") as f:
            f.write(data)
        images.append(name)
    elif ext in VALID_VIDEO:
        with open(path, "wb") as f:
            f.write(data)
        videos.append(name)
    else:
        skipped.append(name)

print(f"\n{'=' * 60}")
print("  РЕЗУЛЬТАТ ЗАГРУЗКИ")
print(f"{'=' * 60}")

if videos:
    print(f"\n  Видео с танцем ({len(videos)}):")
    for f in videos:
        print(f"    {f}")

if images:
    print(f"\n  Фото персонажа ({len(images)}):")
    for f in images:
        print(f"    {f}")

if skipped:
    print(f"\n  Пропущено ({len(skipped)}):")
    for f in skipped:
        print(f"    {f}")

# Готовность
if videos and images:
    print("\n  Готово для SteadyDancer!")
    print("  Выберите эти файлы в ComfyUI:")
    print(f"    VHS_LoadVideo → {videos[0]}")
    print(f"    LoadImage → {images[0]}")
elif videos:
    print("\n  Нужно ещё фото персонажа (.jpg/.png)")
elif images:
    print("\n  Нужно ещё видео с танцем (.mp4)")

# Превью фото
if images:
    print(f"\nПревью:")
    for f in images[:2]:
        path = os.path.join(INPUT_DIR, f)
        try:
            display(IPImage(filename=path, width=300))
        except Exception:
            pass

In [ ]:
#@title 7. Запуск ComfyUI
#@markdown ---
#@markdown ### Способ подключения
#@markdown ngrok — самый стабильный. Cloudflare и localtunnel — без регистрации.
tunnel_method = "ngrok" #@param ["ngrok", "cloudflare", "localtunnel"]
ngrok_token = "" #@param {type:"string"}
#@markdown > Токен ngrok: [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

import subprocess, time, re, os, requests, shutil

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
LOG_FILE = "/content/comfyui.log"

# Убиваем предыдущий экземпляр ComfyUI если есть
!kill -9 $(lsof -t -i:8188) 2>/dev/null || true

comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188",
     "--enable-cors-header", "*"],
    cwd="/content/ComfyUI",
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT
)

print("Запуск ComfyUI...")
started = False
for i in range(24):
    time.sleep(5)
    try:
        if requests.get("http://localhost:8188/api/system_stats", timeout=3).status_code == 200:
            print(f"  ComfyUI запущен за {(i+1)*5} сек!")
            started = True
            break
    except Exception:
        pass

if not started:
    print("  ComfyUI не ответил за 120 сек.")
    with open(LOG_FILE) as f:
        lines = f.readlines()
        errors = [l for l in lines if any(w in l.lower() for w in ['error', 'traceback', 'exception', 'failed'])]
        if errors:
            print("  --- Ошибки ---")
            for l in errors[-15:]:
                print(f"  {l.rstrip()}")
        for l in lines[-10:]:
            print(f"  {l.rstrip()}")
    raise RuntimeError("ComfyUI не запустился. Проверьте ошибки выше.")

with open(LOG_FILE) as f:
    log_text = f.read()
import_errors = [l for l in log_text.split('\n') if 'cannot import' in l.lower() or 'no module named' in l.lower()]
if import_errors:
    print(f"\n  Предупреждения при загрузке нод ({len(import_errors)}):")
    for l in import_errors[:5]:
        print(f"    {l.strip()}")

url = None
if tunnel_method == "ngrok":
    !pip install pyngrok -q
    from pyngrok import ngrok
    if not ngrok_token:
        import getpass
        ngrok_token = getpass.getpass("Введите ngrok authtoken (dashboard.ngrok.com → Your Authtoken): ")
    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(8188)
    url = tunnel.public_url

elif tunnel_method == "cloudflare":
    if not shutil.which("cloudflared"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb 2>&1 | tail -1
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(5)
    for _ in range(20):
        line = tunnel.stdout.readline().decode()
        match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            break

elif tunnel_method == "localtunnel":
    !npm install -g localtunnel > /dev/null 2>&1
    tunnel = subprocess.Popen(
        ["lt", "--port", "8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(8)
    for _ in range(20):
        line = tunnel.stdout.readline().decode()
        match = re.search(r"https://[\w-]+\.loca\.lt", line)
        if match:
            url = match.group(0)
            break

if url:
    print(f"\n{'='*60}")
    print(f"  ComfyUI готов!")
    print(f"  Откройте: {url}")
    print(f"{'='*60}")
    if tunnel_method == "ngrok":
        print(f"\n  ngrok покажет страницу 'Visit Site' при первом входе — нажмите кнопку.")
    elif tunnel_method == "localtunnel":
        print(f"\n  Localtunnel попросит пароль — нажмите ссылку на странице.")
    print(f"\n  1. Меню -> Load -> video_wan_dancer.json")
    print(f"  2. В ноде VHS_LoadVideo выберите видео с танцем")
    print(f"  3. В ноде LoadImage выберите фото персонажа")
    print(f"  4. Нажмите Queue Prompt")
    print(f"\n  Генерация: ~5-10 мин на T4")
    print(f"\n  Если страница белая:")
    print(f"    1. Обновите страницу (Ctrl+R)")
    print(f"    2. Если не помогло — перезапустите с другим tunnel_method")
    print(f"\n  Логи ComfyUI: !cat {LOG_FILE}")
else:
    print(f"\nURL туннеля не найден. Попробуйте другой tunnel_method.")

In [ ]:
#@title 8. Сохранение на Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/dancer"
os.makedirs(dst, exist_ok=True)

results = glob.glob(f"{src}/**/*.mp4", recursive=True)

if not results:
    print("Результатов пока нет. Сначала сгенерируйте видео!")
else:
    for f in results:
        shutil.copy2(f, dst)
        print(f"Скопировано: {os.path.basename(f)}")
    print(f"\n{len(results)} файлов сохранено в {dst}")

---

## Гайд по воркфлоу `video_wan_dancer`

### Основные параметры

| Параметр | Значение | Описание |
|----------|----------|----------|
| `num_frames` | 81 | Количество кадров в выходном видео |
| `steps` | 4 | Шаги генерации (LightX2V LoRA) |
| `CFG` | 1.0 | Для distill LoRA = 1.0 |
| `shift` | 5 | Сдвиг шедулера |
| `block_swap` | 20 | Оффлоад блоков для T4 |
| `context_length` | 81 | Размер контекстного окна |
| `fps` | 30 | Частота кадров выхода |

### Советы для лучшего результата

1. **Видео с танцем:**
   - Один человек в кадре, полная фигура
   - Стабильная камера (без тряски)
   - Хорошее освещение, контрастный фон
   - Длина 5-10 секунд (больше = дольше + больше VRAM)

2. **Фото персонажа:**
   - Полный рост или хотя бы по пояс
   - Нейтральная поза (стоя)
   - Чёткое фото, хорошее освещение
   - Без сложного фона

### Если получается OOM

1. Уменьшите `num_frames` до 49 (меньше кадров)
2. Увеличьте `block_swap` до 30
3. Используйте более короткое видео-источник
4. Перезапустите runtime: Runtime → Restart runtime